[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [24]:

# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        # pass  # Initialize projections
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.num_kv_heads = num_kv_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)

        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        self.repeat_interleave = num_heads // num_kv_heads

    def forward(self, x):
        # pass  # Self-attention with grouped KV
        Q = self.W_q(x) # (B, T, d_model)
        K = self.W_k(x).repeat_interleave(self.repeat_interleave, dim = -1) # (B, T, d_model)
        V = self.W_v(x).repeat_interleave(self.repeat_interleave, dim = -1) # (B, T, d_model)
        score = torch.bmm(K, Q.transpose(-2, -1)) / math.sqrt(self.d_k)
        output = torch.softmax(score, dim=-1).bmm(V)
        print(Q.shape, K.shape, V.shape, output.shape)
        output = self.W_o(output)
        return output



In [25]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)

out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
torch.Size([2, 6, 32]) torch.Size([2, 6, 32]) torch.Size([2, 6, 32]) torch.Size([2, 6, 32])
Output shape: torch.Size([2, 6, 32])


In [26]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 6, 32]) torch.Size([2, 6, 32]) torch.Size([2, 6, 32]) torch.Size([2, 6, 32])
  ✅ [1/5] Output shape (2.6ms)
  ✅ [2/5] nn.Linear with correct shapes (0.5ms)
torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16])
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (4.8ms)
  ✅ [4/5] KV heads are shared correctly (14.5ms)
torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16]) torch.Size([1, 4, 16])
  ✅ [5/5] Gradient flow (45.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (67.5ms total)
  Progress saved. Run status() to see your dashboard.

